<a href="https://colab.research.google.com/github/YUFEIFUT/Demos/blob/main/%E4%BA%BA%E8%84%B8%E8%AF%86%E5%88%ABDemo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 安装 InsightFace 和相关依赖
InsightFace 是目前性能最强的人脸识别开源工具包包之一。我们首先需要安装它以及 ONNX Runtime（用于推理驱动）。

In [ ]:
!pip install -U "numpy<2" insightface onnxruntime-gpu opencv-python

### 初始化 InsightFace 模型
我们将使用 InsightFace 提供的 `buffalo_l` 模型库，这是一个包含人脸检测、识别、对齐等多个任务的大型模型库。

In [ ]:
import cv2
import numpy as np
from insightface.app import FaceAnalysis
from google.colab.patches import cv2_imshow

# 修复 NumPy 2.x/1.24+ 的兼容性问题，因为 insightface 内部使用了 np.int
if not hasattr(np, 'int'):
    np.int = int

# 初始化人脸分析应用
app = FaceAnalysis(name='buffalo_l', providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
app.prepare(ctx_id=0, det_size=(640, 640))

# 读取一张测试图片
img_path = '/content/test1.jpg'
img = cv2.imread(img_path)

if img is not None:
    # 进行人脸检测和特征提取
    faces = app.get(img)

    # 绘制识别结果
    res_img = app.draw_on(img, faces)

    print(f"检测到 {len(faces)} 张人脸")
    cv2_imshow(res_img)
else:
    print("请确保上传了测试图片 /content/test1.jpg 并修改了路径。")

download_path: /root/.insightface/models/buffalo_l


100%|██████████| 281857/281857 [00:03<00:00, 88632.06KB/s]


Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}, 'CUDAExecutionProvider': {'sdpa_kernel': '0', 'use_tf32': '1', 'fuse_conv_bias': '0', 'prefer_nhwc': '0', 'tunable_op_max_tuning_duration_ms': '0', 'enable_skip_layer_norm_strict_mode': '0', 'tunable_op_tuning_enable': '0', 'tunable_op_enable': '0', 'use_ep_level_unified_stream': '0', 'device_id': '0', 'has_user_compute_stream': '0', 'gpu_external_empty_cache': '0', 'cudnn_conv_algo_search': 'EXHAUSTIVE', 'cudnn_conv1d_pad_to_nc1d': '0', 'gpu_mem_limit': '18446744073709551615', 'gpu_external_alloc': '0', 'gpu_external_free': '0', 'arena_extend_strategy': 'kNextPowerOfTwo', 'do_copy_in_default_stream': '1', 'enable_cuda_graph': '0', 'user_compute_stream': '0', 'cudnn_conv_use_max_workspace': '1'}}
find model: /root/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with o

### 测试另一张图片 (test2.jpg)
我们现在尝试对 `/content/test2.jpg` 进行同样的人脸分析。

In [ ]:
img_path2 = '/content/41813孙佳美.jpg'
img2 = cv2.imread(img_path2)

if img2 is not None:
    # 进行人脸检测和特征提取
    faces2 = app.get(img2)

    # 绘制识别结果
    res_img2 = app.draw_on(img2, faces2)

    print(f"检测到 {len(faces2)} 张人脸")
    cv2_imshow(res_img2)
else:
    print("请确保上传了测试图片 /content/test2.jpg 并修改了路径。")

请确保上传了测试图片 /content/test2.jpg 并修改了路径。


### 进阶：构建简单的人脸识别系统
下面的代码演示了如何提取特征向量并进行 1:N 的搜索比对。

In [ ]:
# 1. 模拟一个简单的底库 (Gallery)
face_db = {}

def register_face(name, image_path):
    img = cv2.imread(image_path)
    if img is None: return
    faces = app.get(img)
    if len(faces) > 0:
        # 取检测到的第一张脸的特征
        face_db[name] = faces[0].normed_embedding
        print(f"成功注册: {name}")

# 注册示例 (假设 img2 是孙佳美)
if 'faces2' in locals() and len(faces2) > 0:
    face_db['孙佳美'] = faces2[0].normed_embedding
    print("已从之前的运行结果中注册：孙佳美")

# 2. 定义比对函数
def identify_face(target_embedding, threshold=0.5):
    best_name = "Unknown"
    max_sim = 0

    for name, db_embedding in face_db.items():
        # 计算余弦相似度
        sim = np.dot(target_embedding, db_embedding)
        if sim > max_sim:
            max_sim = sim
            best_name = name

    if max_sim < threshold:
        return "Unknown", max_sim
    return best_name, max_sim

# 3. 测试识别
if 'faces' in locals():
    for i, face in enumerate(faces):
        name, score = identify_face(face.normed_embedding)
        print(f"人脸 {i}: 识别结果 = {name}, 相似度分数 = {score:.4f}")

In [ ]:
# 使用底库图片自身进行测试
test_img_path = '/content/41813孙佳美.jpg'
test_img = cv2.imread(test_img_path)

if test_img is not None:
    test_faces = app.get(test_img)
    print(f"在该图片中检测到 {len(test_faces)} 张人脸")

    for i, face in enumerate(test_faces):
        name, score = identify_face(face.normed_embedding)
        print(f"检测结果: {name}, 相似度: {score:.4f}")

        # 绘制结果展示
        res_img = app.draw_on(test_img, [face])
        cv2_imshow(res_img)
else:
    print("未找到图片，请检查路径。")

未找到图片，请检查路径。


In [ ]:
print("当前底库中注册的人员:")
for name in face_db.keys():
    print(f"- {name}")

if not face_db:
    print("底库目前为空。")

当前底库中注册的人员:
底库目前为空。


In [ ]:
# 注册新的人员，使用根目录路径
# register_face('孙慧敏', '/content/41988孙慧敏.jpg')
# register_face('吴雅婷', '/content/吴雅婷1.jpg')
# register_face('吴倩', '/content/吴倩.jpg')

register_face('孙珍妮', '/content/孙珍妮.jpg')

In [ ]:
# 打印最新的底库名单
print("\n更新后的底库人员:")
for name in face_db.keys():
    print(f"- {name}")


更新后的底库人员:


In [ ]:
# 使用孙佳美的图片作为输入，对比底库中的所有人
test_img_path = '/content/孙珍妮2.jpg'
test_img = cv2.imread(test_img_path)

if test_img is not None:
    faces = app.get(test_img)
    if len(faces) > 0:
        target_embedding = faces[0].normed_embedding
        print(f"输入图片：{test_img_path}\n")
        print("--- 相似度比对结果 ---")

        # 遍历底库进行对比
        for name, db_embedding in face_db.items():
            sim = np.dot(target_embedding, db_embedding)
            print(f"与库中人员 [{name}] 的相似度: {sim:.4f}")
    else:
        print("未在输入图片中检测到人脸。")
else:
    print(f"找不到图片: {test_img_path}")

找不到图片: /content/孙珍妮2.jpg


### 使用 FAISS 加速向量检索
当底库变大时，我们可以将 `face_db` 中的向量构建成一个 FAISS 索引。这样搜索时间会从 $O(N)$ 降到几乎恒定的时间。

In [ ]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 85.5 MB/s eta 0:00:00


In [ ]:
import faiss

# 1. 准备数据
names = list(face_db.keys())
embeddings = np.array(list(face_db.values())).astype('float32')

# 2. 创建 FAISS 索引 (使用内积索引 IndexFlatIP，因为我们的向量已归一化，内积等同于余弦相似度)
dimension = embeddings.shape[1]  # 512
index = faiss.IndexFlatIP(dimension)

# 3. 将底库向量添加到索引中
index.add(embeddings)

print(f"FAISS 索引构建完成，底库包含 {index.ntotal} 个特征向量。")

def search_faiss(query_embedding, top_k=1):
    # FAISS 搜索输入需要是 (1, 512) 的矩阵
    query_embedding = query_embedding.reshape(1, -1).astype('float32')

    # 搜索最相似的 k 个结果
    distances, indices = index.search(query_embedding, top_k)

    results = []
    for i in range(top_k):
        idx = indices[0][i]
        if idx != -1: # -1 表示未找到足够结果
            results.append({
                'name': names[idx],
                'similarity': float(distances[0][i])
            })
    return results

# 4. 测试搜索速度
if 'target_embedding' in locals():
    match = search_faiss(target_embedding, top_k=1)[0]
    print(f"FAISS 检索结果: {match['name']}, 相似度: {match['similarity']:.4f}")

FAISS 索引构建完成，底库包含 6 个特征向量。
FAISS 检索结果: 孙珍妮, 相似度: 0.7690


# 创建gradio界面

## 使用 index 作为内存数据库
支持界面上传人脸以及拍照，以及人脸识别

In [ ]:
!pip install -q gradio

In [ ]:
import gradio as gr
import numpy as np
import cv2

def gradio_register(name, image):
    if image is None or name == "":
        return "请输入姓名并上传图片"

    # Gradio 传入的是 RGB，InsightFace 通常需要 BGR
    img_bgr = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    faces = app.get(img_bgr)

    if len(faces) == 0:
        return "未检测到人脸，请重试"

    # 提取特征并归一化
    embedding = faces[0].normed_embedding.astype('float32')

    # 更新本地 face_db
    face_db[name] = embedding

    # 更新 FAISS 索引
    global index, names
    names.append(name)
    index.add(embedding.reshape(1, -1))

    print(f"[系统日志] 用户上传并注册了新人员: {name}")
    return f"成功注册: {name} (当前底库人数: {len(names)})"

def gradio_identify(image):
    if image is None:
        return "请上传图片", None

    img_bgr = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    faces = app.get(img_bgr)

    if len(faces) == 0:
        return "未检测到人脸", None

    # 获取第一个检测到的人脸进行检索
    query_vec = faces[0].normed_embedding
    match_results = search_faiss(query_vec, top_k=1)

    if not match_results:
        return "底库为空", None

    best_match = match_results[0]
    # 绘制结果图片
    res_img = app.draw_on(img_bgr, [faces[0]])
    res_img_rgb = cv2.cvtColor(res_img, cv2.COLOR_BGR2RGB)

    result_text = f"识别结果: {best_match['name']}\n相似度: {best_match['similarity']:.4f}"
    return result_text, res_img_rgb

# 构建界面
with gr.Blocks() as demo:
    gr.Markdown("## 🚀 InsightFace + FAISS 人脸管理系统")

    with gr.Tab("人脸注册"):
        with gr.Row():
            reg_input_name = gr.Textbox(label="人员姓名")
            reg_input_img = gr.Image(label="上传人脸照片")
        reg_btn = gr.Button("提交注册")
        reg_output = gr.Textbox(label="状态")
        reg_btn.click(gradio_register, inputs=[reg_input_name, reg_input_img], outputs=reg_output)

    with gr.Tab("人脸识别"):
        with gr.Row():
            ident_input_img = gr.Image(label="上传待识别照片")
            ident_output_img = gr.Image(label="检测结果")
        ident_output_text = gr.Textbox(label="识别信息")
        ident_btn = gr.Button("开始识别")
        ident_btn.click(gradio_identify, inputs=ident_input_img, outputs=[ident_output_text, ident_output_img])

demo.launch(share=True, debug=True)

## 使用 supabase 作为数据库

In [ ]:
!pip install -q supabase

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 5.4 MB/s eta 0:00:00


In [ ]:
from supabase import create_client, Client

SUPABASE_URL = "https://cgitldeegnsfyzmubcik.supabase.co"
SUPABASE_KEY = "sb_publishable_Tb-RgkJudeaPn1rbPsDMWQ__NdXAK0v"

supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)

print("Supabase 客户端初始化成功！")

Supabase 客户端初始化成功！


### ⚠️ 重要步骤：在 Supabase SQL Editor 中执行

由于 Python SDK 无法直接修改数据库 Schema，请登录你的 Supabase 后台，打开 **SQL Editor**，运行以下命令来创建支持向量搜索的表：

```sql
-- 1. 开启向量扩展
create extension if not exists vector;

-- 2. 创建人脸底库表
create table if not exists face_profiles (
  id bigserial primary key,
  name text not null,
  embedding vector(512) -- InsightFace buffalo_l 输出的是 512 维向量
);

-- 3. 创建相似度搜索函数 (供 API 调用)
create or replace function match_faces (
  query_embedding vector(512),
  match_threshold float,
  match_count int
)
returns table (
  id bigint,
  name text,
  similarity float
)
language plpgsql
as $$
begin
  return query
  select
    face_profiles.id,
    face_profiles.name,
    1 - (face_profiles.embedding <=> query_embedding) as similarity
  from face_profiles
  where 1 - (face_profiles.embedding <=> query_embedding) > match_threshold
  order by face_profiles.embedding <=> query_embedding
  limit match_count;
end;
$$;
```

In [ ]:
def sync_local_to_supabase():
    """将本地 face_db 的数据批量同步到 Supabase"""
    data_to_insert = []
    for name, embedding in face_db.items():
        # 转换 embedding 为 list 格式以适配 JSON
        data_to_insert.append({
            "name": name,
            "embedding": embedding.tolist()
        })

    if data_to_insert:
        try:
            response = supabase.table("face_profiles").upsert(data_to_insert).execute()
            print(f"成功同步 {len(data_to_insert)} 条记录到 Supabase")
        except Exception as e:
            print(f"同步失败: {e}")
    else:
        print("本地底库为空，无需同步")

# 执行同步
sync_local_to_supabase()

本地底库为空，无需同步


In [ ]:
import gradio as gr
import numpy as np
import cv2

def supabase_register(name, image):
    if image is None or name == "":
        return "请输入姓名并上传图片"

    # Gradio 传入的是 RGB，InsightFace 通常需要 BGR
    img_bgr = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    faces = app.get(img_bgr)

    if len(faces) == 0:
        return "未检测到人脸，请重试"

    # 提取特征并转换为 list 以适配 Supabase vector 类型
    embedding = faces[0].normed_embedding.tolist()

    try:
        # 将数据插入 Supabase
        data = {"name": name, "embedding": embedding}
        supabase.table("face_profiles").insert(data).execute()

        print(f"[系统日志 - Supabase] 用户上传并注册了新人员: {name}")
        return f"成功注册到云端: {name}"
    except Exception as e:
        return f"注册失败: {str(e)}"

def supabase_identify(image):
    if image is None:
        return "请上传图片", None

    img_bgr = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    faces = app.get(img_bgr)

    if len(faces) == 0:
        return "未检测到人脸", None

    # 提取查询向量
    query_vec = faces[0].normed_embedding.tolist()

    try:
        # 调用 Supabase 中定义的 match_faces RPC 函数
        # match_threshold 设置为 0.5 (余弦相似度)
        rpc_params = {
            "query_embedding": query_vec,
            "match_threshold": 0.5,
            "match_count": 1
        }
        response = supabase.rpc("match_faces", rpc_params).execute()
        results = response.data

        if not results:
            return "未在数据库中找到匹配人员", None

        best_match = results[0]

        # 绘制识别结果
        res_img = app.draw_on(img_bgr, [faces[0]])
        res_img_rgb = cv2.cvtColor(res_img, cv2.COLOR_BGR2RGB)

        result_text = f"识别结果 (来自 Supabase): {best_match['name']}\n相似度: {best_match['similarity']:.4f}"
        print(result_text)
        return result_text, res_img_rgb

    except Exception as e:
        return f"检索失败: {str(e)}", None

# 构建新界面
with gr.Blocks() as supabase_demo:
    gr.Markdown("## ☁️ InsightFace + Supabase 云端人脸识别系统")

    with gr.Tab("人脸注册 (存入云端)"):
        with gr.Row():
            reg_input_name = gr.Textbox(label="人员姓名")
            reg_input_img = gr.Image(label="上传人脸照片")
        reg_btn = gr.Button("提交注册")
        reg_output = gr.Textbox(label="状态")
        reg_btn.click(supabase_register, inputs=[reg_input_name, reg_input_img], outputs=reg_output)

    with gr.Tab("人脸识别 (云端检索)"):
        with gr.Row():
            ident_input_img = gr.Image(label="上传待识别照片")
            ident_output_img = gr.Image(label="检测结果")
        ident_output_text = gr.Textbox(label="识别信息")
        ident_btn = gr.Button("开始识别")
        ident_btn.click(supabase_identify, inputs=ident_input_img, outputs=[ident_output_text, ident_output_img])

# 启动界面
supabase_demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://d0ac20627aca5cf38d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.12/dist-packages/insightface/utils/transform.py:68: FutureWarning: `rcond` parameter will change to the default of machine precision times ``max(M, N)`` where M and N are the input matrix dimensions.
To use the future default and silence this warning we advise to pass `rcond=None`, to keep using the old, explicitly pass `rcond=-1`.
  P = np.linalg.lstsq(X_homo, Y)[0].T # Affine matrix. 3 x 4


[系统日志 - Supabase] 用户上传并注册了新人员: 吴雅婷


/usr/local/lib/python3.12/dist-packages/insightface/utils/transform.py:68: FutureWarning: `rcond` parameter will change to the default of machine precision times ``max(M, N)`` where M and N are the input matrix dimensions.
To use the future default and silence this warning we advise to pass `rcond=None`, to keep using the old, explicitly pass `rcond=-1`.
  P = np.linalg.lstsq(X_homo, Y)[0].T # Affine matrix. 3 x 4


识别结果 (来自 Supabase): 吴雅婷
相似度: 1.0000


...

识别结果 (来自 Supabase): 孙慧敏
相似度: 1.0000


/usr/local/lib/python3.12/dist-packages/insightface/utils/transform.py:68: FutureWarning: `rcond` parameter will change to the default of machine precision times ``max(M, N)`` where M and N are the input matrix dimensions.
To use the future default and silence this warning we advise to pass `rcond=None`, to keep using the old, explicitly pass `rcond=-1`.
  P = np.linalg.lstsq(X_homo, Y)[0].T # Affine matrix. 3 x 4
/usr/local/lib/python3.12/dist-packages/insightface/utils/transform.py:68: FutureWarning: `rcond` parameter will change to the default of machine precision times ``max(M, N)`` where M and N are the input matrix dimensions.
To use the future default and silence this warning we advise to pass `rcond=None`, to keep using the old, explicitly pass `rcond=-1`.
  P = np.linalg.lstsq(X_homo, Y)[0].T # Affine matrix. 3 x 4


识别结果 (来自 Supabase): 天美
相似度: 1.0000


/usr/local/lib/python3.12/dist-packages/insightface/utils/transform.py:68: FutureWarning: `rcond` parameter will change to the default of machine precision times ``max(M, N)`` where M and N are the input matrix dimensions.
To use the future default and silence this warning we advise to pass `rcond=None`, to keep using the old, explicitly pass `rcond=-1`.
  P = np.linalg.lstsq(X_homo, Y)[0].T # Affine matrix. 3 x 4


识别结果 (来自 Supabase): 小美
相似度: 1.0000
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://d0ac20627aca5cf38d.gradio.live


In [ ]:
# vector = [-0.09725128,-0.06588039,0.08424267,0.025592348,-0.015389512,-0.03213598,-0.01682402,-0.021183603,-0.024391884,-0.029090239,-0.1057074,0.082866,-0.07545146,-0.023721972,0.049321,-0.09899712,-0.027170287,-0.08505706,-0.03516619,0.04570623,-0.0013474948,0.008933177,-0.026848951,0.03753866,0.0053919004,-0.07213524,-0.046111636,-0.0415867,-0.012032946,-0.014199084,-0.03201976,0.033551328,0.043403827,-0.05851228,-0.015051942,0.019356558,-0.037174605,0.0022535038,0.095628746,0.045804027,-0.03539519,0.106064826,0.0068493923,0.054277383,-0.014431795,0.010918949,-0.009745372,-0.020534676,-0.037833832,-0.037307553,0.017522277,0.0363617,0.015131688,-0.038222615,0.039915103,0.026836244,0.012628117,-0.037315723,0.07576309,-0.0138053335,0.033319738,-0.0067759426,0.0051452103,8.317038e-05,0.0014275397,-0.004184589,0.008784753,-0.03177827,0.0090146065,-0.074681744,0.09500348,-0.032925878,0.058516778,0.025696719,0.04808128,0.01844992,0.015017197,0.014490318,0.013029708,-0.045156084,-0.05563421,-0.029144678,-0.033782374,0.01350859,0.056642167,-0.07293421,-0.07731273,0.029322412,-0.03779158,0.019408152,0.05743045,-0.06838502,0.0028313012,-0.027601583,-0.0056793517,-0.03405526,-0.015168276,0.027093342,0.08603902,0.07262081,0.0353915,-0.04070856,-0.00036070976,-0.015786957,-0.01223909,0.041315284,-0.01651927,-0.060362544,0.028624797,0.063537836,0.016328696,-0.018695494,0.01577409,-0.007001653,-0.039447404,-0.09415106,0.095100135,0.009953094,-0.019532,0.020263808,-0.00033065653,-0.018094128,0.035471313,-0.030227441,0.005169801,0.05786433,0.056243576,0.013177322,0.05829103,-0.03357923,-0.048918087,-0.11160032,0.07520166,0.0638397,-0.05277885,0.0016991398,-0.0050864816,-0.029583449,0.069414295,0.05271619,0.052615393,-0.028617868,0.04536871,0.0051475945,-0.035651624,0.024768475,-0.02004903,0.07919211,-0.0065650274,0.052575707,-0.0043950975,-0.038365144,0.0029820206,0.041270982,-0.035815764,-0.016567754,0.054797843,-0.0017669287,0.0063990345,-0.009721498,-0.033508454,-0.0040672063,0.013142506,0.03773462,-0.027080214,0.028993206,-0.028212657,0.013565627,-0.020820422,0.0073272884,-0.01423959,0.012626661,-0.0432159,0.025304819,0.0435464,-0.028846778,0.017564911,-0.033535358,-0.06802646,-0.01900022,0.029195912,0.013312386,0.019069862,-0.043457832,0.02345415,-0.10394079,0.031399302,0.023235634,0.077891305,0.045401532,-0.04064616,-0.0334815,0.014042505,-0.024351541,-0.036976866,0.066513054,-0.0258229,-0.035423793,-0.05649823,-0.031706925,-0.035737347,-0.07110035,-0.12737335,-0.01774214,-0.015806375,-0.0031306157,0.06686769,-0.011073297,-0.055373505,0.016049137,-0.018696092,0.04065408,0.046666175,0.026728602,0.020598715,0.073614046,0.036636397,0.036185037,-0.029759252,-0.030590348,0.007900115,0.024245545,0.024474392,-0.025065828,-0.020368839,-0.13718064,-0.06451394,-0.022094445,-0.019505253,-0.053822838,-0.024411818,0.020465486,-0.0046480405,-0.011194302,-0.00342462,-0.0579792,0.03186208,-0.0093610035,0.0052147047,-0.03399933,0.04475867,-0.019662397,0.023015961,0.049982145,0.042735286,-0.07463742,0.008506187,0.00681313,-0.007895913,-0.06135764,0.0060032313,-0.045434836,0.008501652,0.009126213,-0.0075981845,-0.018840589,0.025561599,0.0141397845,0.026682222,0.0056907805,0.05034842,-0.008429523,0.055158272,-0.038263522,0.0067009423,-0.03575356,-0.023650553,-0.06600057,-0.01698404,0.03237432,-0.061973136,-0.040119268,-0.021667084,-0.056334496,-0.071704544,0.08569445,0.047308177,0.019116795,0.014150835,0.06883093,0.009928732,0.027670758,-0.05628237,0.053014863,0.00042726283,0.010981597,0.059434928,-0.038365718,0.046206012,-0.0076998305,0.0044497973,0.06768066,-0.081279516,-0.020643374,-0.036374412,0.036489595,-0.036913715,-0.04372383,-0.018431386,0.06773714,0.010781858,0.034957558,-0.009495448,0.01821149,-0.0012224579,-0.084114276,0.0002496232,0.020821715,-0.0074448027,-0.09474471,0.011021547,0.032889508,0.020351302,-0.011484092,-0.02984774,0.040537406,0.016614815,0.06782583,0.026879724,0.05654121,0.011699854,-0.06932528,-0.03822644,-0.019446187,-0.048532918,-0.046515595,0.013479751,0.06322624,-0.026798742,0.046275318,-0.014543425,0.018363815,0.003747428,0.0057613435,0.045323584,-0.04172556,-0.029096596,0.014694608,-0.010194661,0.06781899,-0.05592204,0.005988895,-0.059203707,-0.032611504,0.063104406,0.020012183,0.0016083479,-0.031468146,-0.0057916483,-0.024496036,0.066834085,0.00812334,-0.012546185,-0.053632464,-0.0021105777,-0.057764966,0.04032043,-0.052029602,-0.08545731,0.051136702,0.019297766,0.057273977,-0.05828725,0.065310605,-0.039061643,0.015585065,-0.06268127,-0.039873928,-0.0028158268,-0.01589185,-0.037562266,-0.058808252,0.054308157,-0.013063733,0.01232517,0.014967388,0.041555915,-0.003578185,-0.016539214,0.052135203,3.0361225e-05,0.020625768,0.030565102,-0.052021336,-0.007376278,-0.07255008,0.053811565,-0.03633129,-0.03766347,-0.01606201,-0.02491518,-0.0194367,0.008019049,0.093230724,-0.057606548,0.012146355,0.05450351,-0.037805043,-0.003982794,-0.05638354,-0.12838204,-0.04176567,0.032478284,0.029117474,0.039425008,0.118915975,0.05634334,0.032671485,0.045898866,-0.061899975,-0.04872875,0.01410899,0.0059692045,0.015368046,-0.0063529965,-0.012614393,-0.0043513128,-0.04540699,-0.051112104,-0.06612919,0.0019352627,-0.027630875,0.055382155,0.040844806,0.010806337,0.0018760484,0.032756135,0.0042511607,-1.4902225e-05,0.022983333,-0.06519362,-0.023900801,0.025130183,0.040706344,0.057274837,0.10307534,-0.062319383,-0.08219264,-0.054883067,0.041241538,0.0550338,-0.027805693,-0.023008324,0.15280342,-0.019141056,-0.008150759,-0.095802695,0.06333171,2.1099388e-05,-0.0059995377,0.0051746545,0.026444374,-0.026622914,0.016717672,0.025420902,0.015431797,-0.009694636,-0.038954582,0.013263904,0.023915658,-0.03913133,0.07912754,0.013009009,-0.023331134,-0.011331588,-0.0537879,0.008815528,-0.017468551,-0.014178209,0.016149214,0.023726275,-0.06306879,0.0800946,0.031888112,0.0039830217,-0.00856277,0.06801045,-0.050130542,0.021868967,0.06159214,-0.06820979,0.037849814,0.15069371,0.020727772,0.124145575,0.03429837,-0.004697217,-0.061012138,-0.11377117,-0.06348114,-0.0097163925,0.025401022,0.060417786,0.010078104,0.011938597,0.06292602,-0.0057755294,-0.009571913,-0.014524621,0.028778726,0.0069205402,0.041577622,-0.05460932,-0.004963689,-0.0054652747,0.016400896,0.02586199,0.0014418411,0.013476124,0.05315457,-0.034195144,0.037835203]

# print(f"该向量的维度是: {len(vector)}")

该向量的维度是: 512
